# Task 4: In-Memory Hierarchical Navigable Small World (HNSW) Vector Indexing from Scratch

## Overview
Build an in-memory multi-layer graph index for fast vector search using Cosine Similarity.


In [1]:
import numpy as np
import scipy.spatial.distance as distance
import networkx as nx

# In-memory HNSW vector search graph index
class HNSWIndex:
    def __init__(self, dim=16, M=8):
        self.dim = dim
        self.M = M
        self.vectors = []
        self.graph = nx.Graph()

    def insert(self, vec):
        nid = len(self.vectors)
        self.vectors.append(vec)
        self.graph.add_node(nid)
        if nid > 0:
            dists = [distance.cosine(vec, v) for v in self.vectors[:-1]]
            nearest = np.argsort(dists)[:min(self.M, nid)]
            for n in nearest:
                self.graph.add_edge(nid, n)

    def search(self, query, k=3):
        dists = [distance.cosine(query, v) for v in self.vectors]
        sorted_indices = np.argsort(dists)[:k]
        return [(idx, 1.0 - dists[idx]) for idx in sorted_indices]


In [2]:
# Populate HNSW index with a mini corpus of document vectors and query top matches
doc_corpus = [
    "Quantum computing hardware breakthroughs",
    "Superconducting processor qubits",
    "Algorithms for scientific computing",
    "Deep learning transformer architectures",
    "Natural language processing embeddings"
]

index = HNSWIndex(dim=8)
np.random.seed(42)
for doc in doc_corpus:
    index.insert(np.random.randn(8))

query_vector = np.random.randn(8)
results = index.search(query_vector, k=3)

print("Indexed Document Corpus Count:", len(doc_corpus))
print("Query Search Results (Top 3 Nearest Neighbors):")
for rank, (doc_id, sim) in enumerate(results, 1):
    print(f"  Rank {rank}: Doc #{doc_id} ('{doc_corpus[doc_id]}') - Cosine Similarity: {sim:.4f}")


Indexed Document Corpus Count: 5
Query Search Results (Top 3 Nearest Neighbors):
  Rank 1: Doc #3 ('Deep learning transformer architectures') - Cosine Similarity: 0.5580
  Rank 2: Doc #4 ('Natural language processing embeddings') - Cosine Similarity: 0.3080
  Rank 3: Doc #1 ('Superconducting processor qubits') - Cosine Similarity: 0.1894
